# Training a Transofmer based model with pl lighting

In [1]:
# Add import 
import sys
import torch 
from torch import nn
from torch import optim
from prodigyopt import Prodigy # proddigy optimizer from https://github.com/konstmish/prodigy?tab=readme-ov-file
import pytorch_lightning as pl
from torch.utils.data import DataLoader
import tqdm
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# allow reload of python modules
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
from dataset.RobotPathDataset.normalizer import MinMaxFeatureNormalizer

from model.models import EMA
import copy
import time

# remove all warnings
import warnings
warnings.filterwarnings("ignore")
from dataset import RobotPathDataset

Matplotlib - Backend: 'headless' mode detected -> use 'Agg'


In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
device

cuda:0


device(type='cuda', index=0)

In [3]:
normalizer = MinMaxFeatureNormalizer()
encoder_normalizer = MinMaxFeatureNormalizer()

In [4]:
epochs = 1000
representiation_dim = 64
EXPERIMENT_NAME='100 world 1000 path overfit offset-encoding end2end encoder trianing'

In [5]:

CONFIG = {
    # Model configuration
    'representation_dim': representiation_dim, # Dimension of the representation
    'dim_mults':(1, 2, 4, 8), # Dimension multipliers for hidden layers of the model # (1, 4, 8)
    'attention':False, 
    
    
    
    # Encoder configuration
    'embeder_num_of_hidden_layers' : 1,
    'encoder_conditional_dim': representiation_dim//2,
    
    # Hyperparameters for training
    'batch_size': 8,
    'num_epochs': epochs,
    'ema': EMA(beta=0.99), # Exponential moving average for the model weights
        ## Optimizer configuration
        'optimizer': Prodigy,
        'optimizer_kwargs': {
            'lr': 1., # ! ONLY FOR PRODIGY OPTIMIZER
            'weight_decay': 0.01, 
            'safeguard_warmup':True,
            'use_bias_correction':True,
            'betas': (0.9, 0.99),
            },
        # Scheduler configuration
        'scheduler': torch.optim.lr_scheduler.CosineAnnealingLR,
        'scheduler_kwargs': {
            # 'gamma': 0.999,
            'T_max': 10, # Total number of iterations
        },
    
    'loss_weights':{
        'MSE': 0.3,
        'ObstacleFreePathLoss': 0.5,
        'GraphBasedConsistencyLoss':0.2,
    },
    # Hyperparameters for diffusion process
    'noise_steps': 256,
    'normalize': True,
    'normalizer':normalizer,
    'encoder_normalizer':encoder_normalizer,
    'cfg_scale': 3,


    # Dataset specific configuration
    'n_paths_per_world': 10,
    'n_worlds': 10,
    'n_waypoints': 64, # due to the archtechture has to be a number that is a power of 2 
    'single_world_dataset':False,
}

In [6]:
file = '/home/karim.samir.lotfy/tum-adlr-ss24-09/data/RobotPathData/SingleSphere02_all.db'
# file = '/home/karim.samir.lotfy/tum-adlr-ss24-09/data/SingleSphere02_one-world.db'
dataset = RobotPathDataset(file, n_paths_per_world=CONFIG['n_paths_per_world'], n_worlds=CONFIG['n_worlds'],  n_waypoints=CONFIG['n_waypoints'],   normalizer=CONFIG['normalizer'], single_world_dataset=CONFIG['single_world_dataset'])
print(f' sample shape {dataset[0]["path"].shape} with {len(dataset)} samples consisting of n worlds {len(np.unique(dataset.worlds_indx))}')
indx_sample = 5
sample = dataset[indx_sample];

data shape: torch.Size([100, 64, 2]), min_values: tensor([0, 0], device='cuda:0'), max_values: tensor([10, 10], device='cuda:0'), n_worlds:5, samples: 100
 sample shape torch.Size([64, 2]) with 100 samples consisting of n worlds 5


In [7]:
import torch.utils
import torch.utils.data


class RobotPathDataModule(pl.LightningDataModule):
    def __init__(self, dataset, batch_size: int = 64, val_dataset_ratio=0.01):
        super().__init__()
        self.dataset = dataset
        self.batch_size = batch_size
        self.val_dataset_ratio = val_dataset_ratio

    def setup(self, stage=None):
        # Assign train/val datasets for use in dataloaders
        if stage == 'fit' or stage is None:
            self.train_dataset, self.val_dataset = self._split_train_val(self.dataset)

        # Assign test dataset for use in dataloader(s)
        if stage == 'test' or stage is None:
            assert NotImplementedError('Test dataset is not implemented yet') 

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)


    def _split_train_val(self, dataset):
        return torch.utils.data.random_split(dataset, [1-self.val_dataset_ratio, self.val_dataset_ratio])
    
    def single_sample(self, indx=None):
        if indx is None:
            indx = np.random.randint(0, len(self.dataset))
        return self.dataset[indx]
    
    def single_batch(self):
        return next(iter(self.train_dataloader()))
    
data = RobotPathDataModule(dataset, batch_size=CONFIG['batch_size'], val_dataset_ratio=0.01)
data.setup()

In [8]:
sample = data.single_sample()
sample.keys()

dict_keys(['path', 'og_path', 'world_indx', 'world_img', 'world_distance_field_img', 'offset_path', 'straight_line_path'])

In [35]:
# sin activation

class Sine(nn.Module): # Sine activation
    def __init__(self, w0 = 1.):
        super().__init__()
        self.w0 = w0
    def forward(self, x):
        return torch.sin(self.w0 * x)

class TransformerEncoder(nn.Module):
    def __init__(self, input_dim, output_dim, cond_dim ,d_model, nhead, num_layers, use_sin_activation=False):
        super(TransformerEncoder, self).__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers
         
        self.process_input = nn.Sequential(
            nn.Linear(input_dim, d_model),
            Sine() if use_sin_activation else nn.ReLU(),
        )
        
        self.timestep_encoder = nn.Sequential(
            nn.Linear(1, d_model),
            Sine() if use_sin_activation else nn.ReLU(),
        )
        
        self.cond_encoder = nn.Sequential(
            nn.Linear(cond_dim, d_model),
            Sine() if use_sin_activation else nn.ReLU(),
        )
        
        self.cond_norm = nn.LayerNorm(d_model)
        

        seqTransEncoderLayer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
        self.seqTransformer1 = nn.TransformerEncoder(seqTransEncoderLayer,num_layers=num_layers)
        self.seqTransformer2 = nn.TransformerEncoder(seqTransEncoderLayer,num_layers=num_layers)

        self.proces_output = nn.Sequential(
            nn.Linear(d_model, output_dim),
            # No activation layer as output is gusassian noise anyway
        )
    def forward(self, x, t,cond=None):
        x = self.process_input(x)
        x = self.seqTransformer1(x)
        emb = self.timestep_encoder(t)
        if cond is not None:
            cond = self.cond_encoder(cond)
            emb = emb + cond
        emb = self.cond_norm(emb)
        x = x + emb
        x = self.seqTransformer2(x)
        x = self.proces_output(x)
        return x


In [39]:
from model.diffusion import Diffusion
import pytorch_lightning as L

class PathDiffusionModel(L.LightningModule):
    def __init__(self, config_dict, sample_input_batch, path_type='offset_path', cond_type='world_distance_field_img'):
        super().__init__()
        
        
        self.transition_dim = sample_input_batch[path_type].shape[-1]
        self.representation_dim = config_dict['representation_dim']
        self.path_type = path_type
        self.cond_type = cond_type
        
        # Create model
        self.model = TransformerEncoder(input_dim=self.transition_dim, output_dim=self.transition_dim, cond_dim=representiation_dim, d_model=self.representation_dim, nhead=CONFIG['num_heads'], num_layers=CONFIG['num_layers'], use_sin_activation=CONFIG['use_sin_activation'])

        self.diffusion = Diffusion(input_shape=sample_input_batch.shape[1:], noise_steps=CONFIG['noise_steps'])

        self.save_hyperparameters()
    
    def training_step(self, batch, batch_idx):
        x = batch[self.path_type]
        cond = batch[self.cond_type]
        # sample timesteps
        diffusion
        return loss